Placement Predictor

In [1]:
# Placement Predictor Challenge
# Random Forest Optimization Comparison

import time
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import randint

# -------------------------------
# Phase 1: Data Architecture
# -------------------------------

# Generate synthetic dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    weights=[0.9, 0.1],   # 90% placed, 10% unplaced
    flip_y=0.01,
    random_state=42
)

# Optional feature names
feature_names = [
    "CGPA", "Internships", "Backlogs", "AptitudeScore", "CommunicationSkills",
    "Projects", "Attendance", "CodingSkills", "Certifications", "Workshops",
    "Hackathons", "TechnicalTest", "MockInterview", "Leadership", "Teamwork",
    "ProblemSolving", "Discipline", "Confidence", "Extracurricular", "Research"
]

X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="Placed")  # 1 = placed, 0 = unplaced (or vice versa depending on interpretation)

# Split data: 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Feature scaling
# Important: fit only on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape:", X_test_scaled.shape)
print("Training class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))


# -------------------------------
# Phase 2: Baseline & Metric Trap
# -------------------------------

# Baseline model
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_test_scaled)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print("\n--- Baseline Model ---")
print("Accuracy:", round(baseline_accuracy, 4))
print("F1-Score:", round(baseline_f1, 4))
print(classification_report(y_test, y_pred_baseline))

# Parameter grid for GridSearchCV
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

# Grid Search optimizing Accuracy
grid_acc = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

start_acc = time.time()
grid_acc.fit(X_train_scaled, y_train)
end_acc = time.time()

best_model_acc = grid_acc.best_estimator_
y_pred_grid_acc = best_model_acc.predict(X_test_scaled)

grid_acc_test_accuracy = accuracy_score(y_test, y_pred_grid_acc)
grid_acc_test_f1 = f1_score(y_test, y_pred_grid_acc)

print("\n--- Grid Search (Optimizing Accuracy) ---")
print("Best Parameters:", grid_acc.best_params_)
print("CV Best Score (Accuracy):", round(grid_acc.best_score_, 4))
print("Test Accuracy:", round(grid_acc_test_accuracy, 4))
print("Test F1-Score:", round(grid_acc_test_f1, 4))
print("Time Taken:", round(end_acc - start_acc, 4), "seconds")

# Grid Search optimizing F1
grid_f1 = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

start_f1 = time.time()
grid_f1.fit(X_train_scaled, y_train)
end_f1 = time.time()

best_model_f1 = grid_f1.best_estimator_
y_pred_grid_f1 = best_model_f1.predict(X_test_scaled)

grid_f1_test_accuracy = accuracy_score(y_test, y_pred_grid_f1)
grid_f1_test_f1 = f1_score(y_test, y_pred_grid_f1)

print("\n--- Grid Search (Optimizing F1-Score) ---")
print("Best Parameters:", grid_f1.best_params_)
print("CV Best Score (F1):", round(grid_f1.best_score_, 4))
print("Test Accuracy:", round(grid_f1_test_accuracy, 4))
print("Test F1-Score:", round(grid_f1_test_f1, 4))
print("Time Taken:", round(end_f1 - start_f1, 4), "seconds")


# -------------------------------
# Phase 3: Efficiency Warfare
# -------------------------------

# Exhaustive Grid Search timing (use F1 result for comparison)
grid_time = end_f1 - start_f1

# Randomized Search with wider ranges
param_dist = {
    "n_estimators": randint(10, 501),       # 10 to 500
    "max_depth": [None, 5, 10, 15, 20, 25, 30],
    "min_samples_split": randint(2, 21),    # 2 to 20
    "min_samples_leaf": randint(1, 11)      # 1 to 10
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

start_rand = time.time()
random_search.fit(X_train_scaled, y_train)
end_rand = time.time()

best_random_model = random_search.best_estimator_
y_pred_random = best_random_model.predict(X_test_scaled)

random_test_accuracy = accuracy_score(y_test, y_pred_random)
random_test_f1 = f1_score(y_test, y_pred_random)
random_time = end_rand - start_rand

print("\n--- Randomized Search (Optimizing F1-Score) ---")
print("Best Parameters:", random_search.best_params_)
print("CV Best Score (F1):", round(random_search.best_score_, 4))
print("Test Accuracy:", round(random_test_accuracy, 4))
print("Test F1-Score:", round(random_test_f1, 4))
print("Time Taken:", round(random_time, 4), "seconds")


# -------------------------------
# Comparison Table
# -------------------------------

comparison_table = pd.DataFrame({
    "Method": [
        "Baseline Random Forest",
        "Grid Search (Accuracy)",
        "Grid Search (F1)",
        "Randomized Search (F1)"
    ],
    "Best Parameters": [
        "Default",
        str(grid_acc.best_params_),
        str(grid_f1.best_params_),
        str(random_search.best_params_)
    ],
    "Test Accuracy": [
        round(baseline_accuracy, 4),
        round(grid_acc_test_accuracy, 4),
        round(grid_f1_test_accuracy, 4),
        round(random_test_accuracy, 4)
    ],
    "Test F1-Score": [
        round(baseline_f1, 4),
        round(grid_acc_test_f1, 4),
        round(grid_f1_test_f1, 4),
        round(random_test_f1, 4)
    ],
    "Time Taken (sec)": [
        0,
        round(end_acc - start_acc, 4),
        round(end_f1 - start_f1, 4),
        round(random_time, 4)
    ]
})

print("\n--- Final Comparison Table ---")
print(comparison_table)

Training set shape: (800, 20)
Test set shape: (200, 20)
Training class distribution:
 Placed
0    0.8925
1    0.1075
Name: proportion, dtype: float64
Test class distribution:
 Placed
0    0.895
1    0.105
Name: proportion, dtype: float64

--- Baseline Model ---
Accuracy: 0.93
F1-Score: 0.5333
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       179
           1       0.89      0.38      0.53        21

    accuracy                           0.93       200
   macro avg       0.91      0.69      0.75       200
weighted avg       0.93      0.93      0.92       200


--- Grid Search (Optimizing Accuracy) ---
Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
CV Best Score (Accuracy): 0.9138
Test Accuracy: 0.93
Test F1-Score: 0.5333
Time Taken: 15.4628 seconds

--- Grid Search (Optimizing F1-Score) ---
Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
CV Best Score (F1): 0.3291
Tes